# Data Preparation

This notebook prepares the enriched homestay dataset for the recommendation system by:

1. Inspecting and validating the enriched dataset.
2. Handling missing values and data inconsistencies.
3. Creating amenity-based text features from binary amenity indicators.
4. Creating location-based text features from geographical information.
5. Combining descriptions, amenities, and location information into a unified feature representation.
6. Preparing explainability-related features for recommendation interpretation.
7. Saving the processed dataset for recommendation model development.

The prepared dataset will be used for TF-IDF vectorization, similarity computation, and explainable homestay recommendation generation.

In [24]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

In [25]:
# ==========================================
# LOAD ENRICHED DATASET
# ==========================================

df = pd.read_csv(
    "../data/processed/homestays_enriched.csv"
)

print(f"Dataset Shape: {df.shape}")

Dataset Shape: (1157, 35)


In [26]:
# ==========================================
# DATASET INFORMATION
# ==========================================

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1157 entries, 0 to 1156
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   homestay_id             1157 non-null   int64  
 1   homestay_name           1157 non-null   str    
 2   owner_name              1157 non-null   str    
 3   category                1136 non-null   str    
 4   district                1157 non-null   str    
 5   block                   1157 non-null   str    
 6   village                 1157 non-null   str    
 7   owner_email             1157 non-null   str    
 8   owner_mobile            1157 non-null   int64  
 9   google_name             1157 non-null   str    
 10  google_address          1157 non-null   str    
 11  latitude                1157 non-null   float64
 12  longitude               1157 non-null   float64
 13  rating                  1157 non-null   float64
 14  review_count            1157 non-null   int64  
 15

In [27]:
# ==========================================
# REMOVE LEADING/TRAILING SPACES
# ==========================================

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

In [28]:
# ==========================================
# REMOVE NEWLINES / EXTRA WHITESPACES
# ==========================================

text_cols = [
    "homestay_name",
    "owner_name",
    "village",
    "google_address"
]

for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[\r\n]+", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

In [29]:
# ==========================================
# CHECK MISSING VALUES
# ==========================================

missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print("\nMissing Values")
print(missing_summary[missing_summary > 0])


Missing Values
category    21
dtype: int64


In [30]:
# ==========================================
# STANDARDIZE CAPITALIZATION
# ==========================================

title_cols = [
    "homestay_name",
    "owner_name",
    "category",
    "district",
    "block",
    "village"
]

for col in title_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.title()

In [31]:
# ==========================================
# STANDARDIZE BLOCK NAMES
# ==========================================

block_mapping = {
    "Kalimpong 1": "Kalimpong I",
    "Kalimpong-1": "Kalimpong I",
    "Kalimpong-I": "Kalimpong I",
}

df["block"] = (
    df["block"]
    .replace(block_mapping)
)

In [32]:
# ==========================================
# CREATE DUPLICATE CHECK KEY
# ==========================================

df["duplicate_check_key"] = (
    df["homestay_name"].fillna("") + "_" +
    df["village"].fillna("")
)

possible_duplicates = (
    df["duplicate_check_key"]
    .duplicated(keep=False)
    .sum()
)

print("Possible duplicate records:",
      possible_duplicates)

Possible duplicate records: 4


In [33]:
# ==========================================
# REMOVE DUPLICATE RECORDS
# ==========================================

before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"Removed {before-after} exact duplicates")

Removed 0 exact duplicates


In [34]:
# ==========================================
# VALIDATE COORDINATES
# ==========================================

if {"latitude", "longitude"}.issubset(df.columns):

    invalid_lat = (
        (df["latitude"] < -90) |
        (df["latitude"] > 90)
    )

    invalid_lon = (
        (df["longitude"] < -180) |
        (df["longitude"] > 180)
    )

    print(
        "Invalid coordinates:",
        (invalid_lat | invalid_lon).sum()
    )

Invalid coordinates: 0


In [35]:
# ==========================================
# HANDLE RECOMMENDATION MISSING VALUES
# ==========================================

df["category"] = df["category"].fillna("Silver")


In [36]:
# ==========================================
# HANDLE DISTANCE FEATURES
# ==========================================

distance_columns = [
    "distance_to_deolo",
    "distance_to_durpin",
    "distance_to_town",
    "distance_to_lava",
    "distance_to_pedong",
    "distance_to_gorubathan",
    "distance_to_rishop",
    "distance_to_lolegaon"
]

for col in distance_columns:

    df[col] = df[col].fillna(
        df[col].median()
    )

In [37]:
# ==========================================
# HANDLE PROXIMITY FEATURES
# ==========================================

df["deolo_proximity"] = (
    df["deolo_proximity"]
    .fillna("Far")
)

df["durpin_proximity"] = (
    df["durpin_proximity"]
    .fillna("Far")
)

df["town_proximity"] = (
    df["town_proximity"]
    .fillna("Far")
)

In [38]:
# ==========================================
# CREATE PRICE BANDS
# ==========================================

def get_price_band(price):

    if price < 2000:
        return "budget"

    elif price < 3500:
        return "mid_range"

    else:
        return "premium"

df["price_band"] = df["price"].apply(get_price_band)

In [39]:
# ==========================================
# CREATE AMENITY TEXT FEATURE
# ==========================================

def create_amenity_text(row):

    amenities = []

    if row["wifi"] == 1:
        amenities.append("wifi")

    if row["parking"] == 1:
        amenities.append("parking")

    if row["breakfast"] == 1:
        amenities.append("breakfast")

    if row["mountain_view"] == 1:
        amenities.append("mountain_view")

    if row["room_service"] == 1:
        amenities.append("room_service")

    if row["bonfire_barbeque"] == 1:
        amenities.append("bonfire_barbeque")

    if row["pickup_dropoff_service"] == 1:
        amenities.append("pickup_dropoff_service")

    return " ".join(amenities)


df["amenity_text"] = df.apply(
    create_amenity_text,
    axis=1
)

df[[
    "homestay_name",
    "amenity_text"
]].head()

,homestay_name,amenity_text
0,Revere Homestay,parking breakfast
1,Mansarover Homestay,wifi parking bonfire_barbeque
2,Bethany Homestay,parking breakfast mountain_view room_service b...
3,S3 Homestay,parking breakfast mountain_view room_service b...
4,Bajarangi Homestay,wifi parking breakfast mountain_view bonfire_b...


In [40]:
# ==========================================
# CREATE LOCATION TEXT FEATURE
# ==========================================

df["location_text"] = (

    df["district"].astype(str)

    + " "

    + df["block"].astype(str)

    + " "

    + df["village"].astype(str)

)

In [41]:
# ==========================================
# CREATE FEATURE TEXT
# ==========================================

df["feature_text"] = (

    df["description"].astype(str)

    + " "

    + df["amenity_text"].astype(str)

    + " "

    + df["location_text"].astype(str)

    + " "

    + df["category"].astype(str)
    
    + " "

    + df["price_band"].astype(str)

)

In [42]:
# ==========================================
# FEATURE TEXT SAMPLE
# ==========================================

print(
    df.loc[0, "feature_text"]
)

Revere Homestay in 8th Mile, Kalimpong Village offers convenient access to town, Deolo, and Durpin. Enjoy essential amenities including free Wi-Fi, parking, breakfast, mountain views, and pickup/drop-off service. Rated 5.0 with one review, this Silver-category homestay suits travelers seeking proximity to key locations and reliable, well-located accommodations. parking breakfast Kalimpong Municipality 8Th Mile, Kalimpong Silver mid_range


In [43]:
# ==========================================
# EXPLAINABILITY FEATURE SELECTION
# ==========================================

explainability_features = [

    "price",

    "rating",

    "review_count",

    "distance_to_deolo",

    "distance_to_durpin",

    "distance_to_town",

    "distance_to_lava",

    "distance_to_pedong",

    "distance_to_gorubathan",

    "distance_to_rishop",

    "distance_to_lolegaon"

]

print(
    df[explainability_features]
    .head()
)

   price  rating  review_count  distance_to_deolo  distance_to_durpin  \
0   2126     5.0             1               5.17                3.03   
1   3959     4.2            77               5.79                2.01   
2   1860     4.6            69               0.96                7.57   
3   2130     4.3           106               5.35                2.35   
4   2095     4.2             5               9.03                6.63   

   distance_to_town  distance_to_lava  distance_to_pedong  \
0              0.96             18.01               18.75   
1              1.08             18.60               19.30   
2              4.71             12.85               13.54   
3              0.74             18.12               18.81   
4              7.13             19.19               19.62   

   distance_to_gorubathan  distance_to_rishop  distance_to_lolegaon  
0                   26.25               19.75                 11.49  
1                   25.64               19.96         

In [44]:
# ==========================================
# FINAL CLEANUP BEFORE SAVE
# ==========================================

# duplicate_check_key was only needed for the duplicate check above
df = df.drop(columns=["duplicate_check_key"])

print("\nMissing Values:")
print(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)



Missing Values:
homestay_id               0
homestay_name             0
owner_name                0
category                  0
district                  0
block                     0
village                   0
owner_email               0
owner_mobile              0
google_name               0
google_address            0
latitude                  0
longitude                 0
rating                    0
review_count              0
distance_to_deolo         0
distance_to_durpin        0
distance_to_town          0
distance_to_lava          0
distance_to_pedong        0
distance_to_gorubathan    0
distance_to_rishop        0
distance_to_lolegaon      0
deolo_proximity           0
durpin_proximity          0
town_proximity            0
price                     0
wifi                      0
parking                   0
breakfast                 0
mountain_view             0
room_service              0
bonfire_barbeque          0
pickup_dropoff_service    0
description               0
pri

In [46]:
# ==========================================
# SAVE PREPARED DATASET
# ==========================================

df.to_csv(
    "../data/final/homestays_prepared.csv",
    index=False
)

print("Prepared dataset saved successfully.")
print(f"Final Shape: {df.shape}")

Prepared dataset saved successfully.
Final Shape: (1157, 39)
